# FLAG 2027 - Notebook 04: Extract NEW, stronger features from the raw photos and audio

Notebook 03 showed the old features (VGGFace + ECAPA) carry little identity information beyond gender. Here we extract new embeddings:

| Name | Modality | Model | Why |
|---|---|---|---|
| `arcface` | face | ArcFace ResNet-50 (InsightFace, WebFace600K) | modern face-identity model, much stronger than VGGFace (2015) |
| `clip` | face | CLIP ViT-L/14 (OpenAI) | captures age, ethnicity, appearance - cues that link a face to a voice |
| `wavlm_sv` | voice | WavLM-base-plus speaker-verification (Microsoft) | strong speaker embedding; pretrained on ENGLISH only (Bengali rule safe) |

Then a quick check: how much **identity** and how much **gender** each embedding carries (the same probing idea as your thesis).

**Rule we respect:** dev files are only passed through frozen pretrained models - nothing is learned from them.

**How to run:** T4 GPU. Runtime > Run all. About 20-40 min (most of it unzipping). Each feature set is saved to Drive as soon as it is finished; re-running skips finished ones.

**When done:** paste the **SUMMARY** box to Claude.

In [ ]:
# Cell 1 - Drive, GPU, install insightface + GPU onnxruntime
# (insightface pulls in the CPU-only 'onnxruntime'; remove it AFTERWARDS and keep only onnxruntime-gpu)
from google.colab import drive
drive.mount('/content/drive')
!pip -q install insightface
!pip -q uninstall -y onnxruntime onnxruntime-gpu
!pip -q install onnxruntime-gpu
import onnxruntime as ort
print('onnxruntime providers:', ort.get_available_providers())   # should include CUDAExecutionProvider
import torch
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', dev)

In [ ]:
# Cell 2 - unzip EVERYTHING (incl. jpg/wav) to the fast local disk, then list all items
import os, json, time, zipfile, shutil
import numpy as np, pandas as pd

BASE = '/content/drive/MyDrive'
PROJECT = os.path.join(BASE, 'FLAG2027')
ZIP_DIR = json.load(open(os.path.join(PROJECT, 'paths.json')))['ZIP_DIR']
LOCAL = '/content/flag_data'
FEATDIR = os.path.join(PROJECT, 'features'); os.makedirs(FEATDIR, exist_ok=True)
os.makedirs(LOCAL, exist_ok=True)
for z in ['train_set.zip', 'dev_set.zip']:
    flag = os.path.join(LOCAL, z + '.done')
    if os.path.exists(flag):
        continue
    t0 = time.time()
    shutil.copy(os.path.join(ZIP_DIR, z), '/content/' + z)
    with zipfile.ZipFile('/content/' + z) as zf:
        zf.extractall(LOCAL)
    os.remove('/content/' + z)
    open(flag, 'w').close()
    print('%s unzipped in %.0f s' % (z, time.time() - t0))

INDEX = [os.path.join(r, f) for r, d, fs in os.walk(LOCAL) for f in fs if f.endswith('.txt')]
def find_one(*parts):
    hits = [p for p in INDEX if all(x in p for x in parts)]
    assert len(hits) == 1, 'looking for %s -> found %s' % (parts, hits)
    return hits[0]

def resolve(rel, list_path):
    # the paths in the lists are relative; walk up from the list file until the file exists
    p = os.path.dirname(list_path)
    while p.startswith(LOCAL):
        c = os.path.join(p, rel)
        if os.path.exists(c):
            return c
        p = os.path.dirname(p)
    return None

def load_split(list_path, is_train):
    ids, voices, faces = [], [], []
    for l in open(list_path):
        t = l.split()
        if not t:
            continue
        v, f = (t[2], t[3]) if is_train else (t[1], t[2])
        ids.append(t[0]); voices.append(resolve(v, list_path)); faces.append(resolve(f, list_path))
    return dict(ids=ids, voices=voices, faces=faces)

SPLITS = {'train': load_split(find_one('train_English.txt'), True)}
for track in ['no_gender', 'gender']:
    for lang in ['English', 'Bangla']:
        SPLITS[track + '_' + lang] = load_split(find_one('/' + track + '/', lang + '_test.txt'), False)
for k, s in SPLITS.items():
    print('%-18s %5d items | missing voices %d | missing faces %d'
          % (k, len(s['ids']), sum(p is None for p in s['voices']), sum(p is None for p in s['faces'])))

SUMMARY = []
def note(msg):
    print(msg); SUMMARY.append(msg)

In [ ]:
# Cell 3 - look at the data with your own eyes (always do this!)
from PIL import Image
import soundfile as sf
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
pick = rng.choice(len(SPLITS['train']['faces']), 8, replace=False)
fig, axes = plt.subplots(1, 8, figsize=(16, 2.6))
for ax, i in zip(axes, pick):
    im = Image.open(SPLITS['train']['faces'][i])
    ax.imshow(im); ax.set_title('%dx%d' % im.size, fontsize=8); ax.axis('off')
plt.show()
durs = [sf.info(SPLITS['train']['voices'][i]).duration for i in pick]
srs = [sf.info(SPLITS['train']['voices'][i]).samplerate for i in pick]
note('Face image sizes (sample): %s' % sorted(set(Image.open(SPLITS['train']['faces'][i]).size for i in pick)))
note('Audio sample rates: %s | durations (s): %s' % (sorted(set(srs)), [round(d, 1) for d in durs]))

In [ ]:
# Cell 4 - the encoders (each returns one embedding per file)
import librosa, urllib.request, cv2
from tqdm.auto import tqdm

def batched(items, bs):
    for i in range(0, len(items), bs):
        yield items[i:i + bs]

# --- CLIP ViT-L/14 (face appearance / attributes) ---
# v2 fix: call vision tower + projection explicitly (works on every transformers version)
def enc_clip(paths):
    from transformers import CLIPModel, CLIPProcessor
    name = 'openai/clip-vit-large-patch14'
    model = CLIPModel.from_pretrained(name).half().to(dev).eval()
    proc = CLIPProcessor.from_pretrained(name)
    out = []
    for b in tqdm(list(batched(paths, 64)), desc='clip'):
        ims = [Image.open(p).convert('RGB') for p in b]
        x = proc(images=ims, return_tensors='pt')['pixel_values'].to(dev, torch.float16)
        with torch.no_grad():
            pooled = model.vision_model(pixel_values=x).pooler_output
            out.append(model.visual_projection(pooled).float().cpu().numpy())
    return np.concatenate(out)

# --- ArcFace v1 (NO alignment - kept only for comparison; it scored badly) ---
def enc_arcface(paths):
    import onnxruntime as ort
    mdir = os.path.join(PROJECT, 'models_cache', 'buffalo_l')
    onnx_path = os.path.join(mdir, 'w600k_r50.onnx')
    if not os.path.exists(onnx_path):
        os.makedirs(mdir, exist_ok=True)
        urllib.request.urlretrieve('https://github.com/deepinsight/insightface/releases/download/v0.7/buffalo_l.zip', '/content/buffalo_l.zip')
        with zipfile.ZipFile('/content/buffalo_l.zip') as zf:
            member = [m for m in zf.namelist() if m.endswith('w600k_r50.onnx')][0]
            with open(onnx_path, 'wb') as f:
                f.write(zf.read(member))
    sess = ort.InferenceSession(onnx_path, providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
    inp = sess.get_inputs()[0].name
    out = []
    for b in tqdm(list(batched(paths, 64)), desc='arcface'):
        ims = [np.asarray(Image.open(p).convert('RGB').resize((112, 112)), dtype=np.float32) for p in b]
        x = (np.stack(ims).transpose(0, 3, 1, 2) - 127.5) / 127.5
        out.append(sess.run(None, {inp: x.astype(np.float32)})[0])
    return np.concatenate(out)

# --- ArcFace v2: detect face + 5-point alignment (official InsightFace pipeline), then ArcFace R50 ---
def enc_arcface_aligned(paths):
    from insightface.app import FaceAnalysis
    app = FaceAnalysis(name='buffalo_l', root=os.path.join(PROJECT, 'models_cache', 'insightface'),
                       allowed_modules=['detection', 'recognition'],
                       providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
    app.prepare(ctx_id=0, det_size=(320, 320), det_thresh=0.3)
    rec = app.models['recognition']
    out, no_face = [], 0
    for p in tqdm(paths, desc='arcface_aligned'):
        img = cv2.imread(p)                                                # BGR, 224x224 crop
        big = cv2.copyMakeBorder(img, 56, 56, 56, 56, cv2.BORDER_CONSTANT, value=0)   # margin helps the detector
        faces = app.get(big)
        if faces:
            f = max(faces, key=lambda f: (f.bbox[2] - f.bbox[0]) * (f.bbox[3] - f.bbox[1]))
            out.append(f.normed_embedding)
        else:   # fallback: central crop, resized (rare)
            no_face += 1
            h, w = img.shape[:2]; c = int(0.15 * h)
            e = rec.get_feat(cv2.resize(img[c:h - c, c:w - c], (112, 112))).flatten()
            out.append(e / (np.linalg.norm(e) + 1e-9))
    note('arcface_aligned: no face detected in %d of %d images (used fallback)' % (no_face, len(paths)))
    return np.stack(out)

# --- WavLM-base-plus speaker verification x-vector (voice identity) ---
def enc_wavlm_sv(paths, max_sec=10):
    from transformers import AutoFeatureExtractor, WavLMForXVector
    name = 'microsoft/wavlm-base-plus-sv'
    fe = AutoFeatureExtractor.from_pretrained(name)
    model = WavLMForXVector.from_pretrained(name).to(dev).eval()
    out = []
    for b in tqdm(list(batched(paths, 32)), desc='wavlm_sv'):
        wavs = [librosa.load(p, sr=16000, mono=True)[0][:16000 * max_sec] for p in b]
        x = fe(wavs, sampling_rate=16000, padding=True, return_attention_mask=True, return_tensors='pt').to(dev)
        with torch.no_grad():
            out.append(model(**x).embeddings.cpu().numpy())
    return np.concatenate(out)

# finished ones (arcface, wavlm_sv) are skipped automatically in the next cell
ENCODERS = {'arcface': ('faces', enc_arcface), 'arcface_aligned': ('faces', enc_arcface_aligned),
            'clip': ('faces', enc_clip), 'wavlm_sv': ('voices', enc_wavlm_sv)}

In [ ]:
# Cell 5 - extract and save (one .npz per encoder, holding every split; skipped if it already exists)
for name, (kind, fn) in ENCODERS.items():
    out_path = os.path.join(FEATDIR, name + '.npz')
    if os.path.exists(out_path):
        print(name, 'already extracted - skipping'); continue
    t0 = time.time()
    try:
        allpaths = [p for s in SPLITS.values() for p in s[kind]]
        emb = fn(allpaths)
    except Exception as e:
        note('ENCODER %s FAILED: %r' % (name, e)); continue
    saved, i = {}, 0
    for k, s in SPLITS.items():
        n = len(s[kind]); saved[k] = emb[i:i + n].astype(np.float32); i += n
    np.savez(out_path, **saved)
    note('%-9s dim %4d | %d files in %.1f min -> saved' % (name, emb.shape[1], len(allpaths), (time.time() - t0) / 60))

In [ ]:
# Cell 6 - how much IDENTITY and how much GENDER does each embedding carry? (TRAIN data only)
from sklearn.metrics import roc_curve
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def eer(labels, scores):
    fpr, tpr, _ = roc_curve(labels, scores); fnr = 1 - tpr
    i = np.nanargmin(np.abs(fnr - fpr)); return (fpr[i] + fnr[i]) / 2 * 100

# identities + gender for the train rows
train_lines = [l.split() for l in open(find_one('train_English.txt')) if l.strip()]
spk = np.array([t[4] for t in train_lines])
meta = pd.read_csv(os.path.join(ZIP_DIR, 'meta_file_train_set.csv'), header=None, names=['spk', 'g'])
spk2g = dict(zip(meta.spk.astype(str).str.strip(), meta.g.astype(str).str.strip().str.lower()))
gen = np.array([spk2g[s] for s in spk])

def old_feats():
    def rd(name, dim):
        p = [os.path.join(r, f) for r, d, fs in os.walk(LOCAL) for f in fs if f == name][0]
        return pd.read_csv(p, header=None).iloc[:, :dim].values.astype(np.float32)
    return {'vggface (old)': rd('train_English_faces.csv', 4096), 'ecapa (old)': rd('train_English_voices.csv', 192)}

FEATS = old_feats()
for name in ENCODERS:
    p = os.path.join(FEATDIR, name + '.npz')
    if os.path.exists(p):
        FEATS[name] = np.load(p)['train']

rng = np.random.default_rng(0)
people = np.unique(spk); rows = {s: np.where(spk == s)[0] for s in people}
A, B, L = [], [], []
for _ in range(5000):
    s = rng.choice(people); a, b = rng.choice(rows[s], 2, replace=False); A.append(a); B.append(b); L.append(1)
    s, t = rng.choice(people, 2, replace=False); A.append(rng.choice(rows[s])); B.append(rng.choice(rows[t])); L.append(0)
A, B, L = np.array(A), np.array(B), np.array(L)

note('EMBEDDING CHECK (train only) | same-modality identity EER (lower = more identity) | gender probe acc (people-disjoint)')
for name, X in FEATS.items():
    Xn = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
    e = eer(L, (Xn[A] * Xn[B]).sum(1))
    probe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=0.1))
    acc = cross_val_score(probe, X, gen, groups=spk, cv=GroupKFold(5)).mean()
    note('  %-14s dim %4d | identity EER %5.1f%% | gender probe %.2f' % (name, X.shape[1], e, acc))

In [ ]:
# Cell 7 - summary
print('=' * 70)
print('SUMMARY  (copy everything between the lines and paste to Claude)')
print('=' * 70)
for s in SUMMARY:
    print(s)
print('=' * 70)